# Model 3 — Workload Forecasting (ticket volume)

**Question:** how many support tickets will be created per day / week, so managers can see workload spikes coming?

**Pipeline of this notebook**

1. Data and cleaning → daily ticket-count series
2. Exploratory analysis (what patterns can a model use?)
3. Evaluation design (cross-validation + untouched final test)
4. Results: baselines vs tuned candidates
5. Error analysis (where does the model fail?)
6. High-load flag, next-14-day forecast
7. Conclusions and limitations

> Run `python -m ml.forecasting.run_models` (from the repo root) first so that `ml/artifacts/forecasting/` contains the result tables this notebook reads.
> Numbers in the text below are always computed from those files, so they match **your** run.

In [ ]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# find the project root (folder that contains ml/forecasting) so the notebook works from anywhere
ROOT = Path.cwd().resolve()
while not (ROOT / "ml" / "forecasting").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook inside the OpsPilot-AI repo folder.")
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})

from ml.forecasting.prepare_series import (load_issues, build_daily_series, build_holiday_share,
                                            quality_report, SUPPORT_TYPES)
from ml.forecasting.baselines import BASELINES
from ml.forecasting.models import prophet_model, xgboost_model, ensemble
from ml.forecasting.cv import make_folds, cross_validate, walk_forward

ISSUES, HOLI = "data/raw/issues.csv", "data/raw/holidays_by_country.csv"
START, FINAL_TEST, HORIZON, N_FOLDS, FOLD_DAYS = "2018-01-01", 84, 7, 6, 56
OUT = Path("ml/artifacts/forecasting")
print("project root:", ROOT)

## 1. Data and cleaning

`issues.csv` has **one row per ticket**. Forecasting needs **one row per day**, so we count tickets by creation date.

Two cleaning decisions:

* **Only real support work.** The file also contains types such as `Vacation`, `Epic`, `Sprint Summary` and `Retrospective`. They are not customer workload, so they are excluded (`SUPPORT_TYPES`).
* **UTC calendar days, 2018 onward.** Early years look like a different regime (see the chart in section 2). The start date is a parameter (`START`).

In [ ]:
raw = load_issues(ISSUES)
print(f"tickets in file: {len(raw):,}  |  unique ids: {raw['id'].nunique():,}")
print(f"created from {raw['issue_created'].min().date()} to {raw['issue_created'].max().date()}")

types = raw["issue_type"].value_counts()
colors = ["tab:blue" if t in SUPPORT_TYPES else "lightgrey" for t in types.index]
ax = types.plot.barh(color=colors, figsize=(8, 4.5)); ax.invert_yaxis()
ax.set_title("Ticket types (blue = kept as support workload)"); ax.set_xlabel("tickets")
plt.show()

In [ ]:
daily = build_daily_series(raw, start=START)
q = quality_report(daily)
display(pd.DataFrame({"value": {k: v for k, v in q.items() if k != "avg_by_weekday"}}))

## 2. Exploratory analysis

What structure is there for a model to learn?

In [ ]:
# 2a. long-run volume: why do we start in 2018?
support = raw[raw["issue_type"].isin(SUPPORT_TYPES)]
by_year = support.groupby(support["issue_created"].dt.year).size()
cols = ["tab:blue" if y >= pd.Timestamp(START).year else "lightgrey" for y in by_year.index]
ax = by_year.plot.bar(color=cols, figsize=(11, 3.6))
ax.set_title("Support tickets per year (blue = used for modelling)"); ax.set_xlabel(""); ax.set_ylabel("tickets")
plt.show()

In [ ]:
# 2b. day-of-week pattern
names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow = daily.groupby(daily.index.dayofweek).mean()
ax = dow.rename(index=dict(enumerate(names))).plot.bar(
        color=["tab:blue"]*4 + ["lightgrey"]*2 + ["tab:blue"], figsize=(7, 3.6))
ax.set_title("Average tickets per day of week"); ax.set_ylabel("tickets / day"); ax.set_xlabel("")
plt.show()
print("Working week here is Sunday-Thursday. Friday/Saturday are the weekend (is_weekend feature).")

In [ ]:
# 2c. the whole series with a 28-day rolling mean
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily.values, lw=.5, color="tab:blue", alpha=.6, label="daily count")
ax.plot(daily.index, daily.rolling(28).mean(), lw=2, color="black", label="28-day mean")
ax.axvspan(daily.index[-FINAL_TEST], daily.index[-1], color="tab:orange", alpha=.15, label="final test (held out)")
ax.set_title("Daily support tickets"); ax.set_ylabel("tickets / day"); ax.legend(loc="upper left")
plt.show()

In [ ]:
# 2d. is there a yearly (month-of-year) pattern?  Prophet's yearly term was switched off during tuning.
m = daily.resample("MS").mean()
tab = m.groupby([m.index.year, m.index.month]).first().unstack(0)
ax = tab.plot(figsize=(11, 3.8), marker="o", lw=1.2)
ax.set_xticks(range(1, 13)); ax.set_xlabel("month"); ax.set_ylabel("mean tickets / day")
ax.set_title("Monthly average by year (parallel lines would suggest yearly seasonality)")
ax.legend(title="year", ncol=3, fontsize=8)
plt.show()

In [ ]:
# 2e. holiday effect. holiday_share = share of ticket volume coming from countries that are on holiday that day.
test_start = daily.index[-FINAL_TEST]
future_idx = pd.date_range(daily.index[0], daily.index[-1] + pd.Timedelta(days=30), freq="D")
hs, country_weights = build_holiday_share(ISSUES, HOLI, future_idx, weights_until=test_start)

workdays = daily[~daily.index.dayofweek.isin([4, 5])]
heavy = hs.reindex(workdays.index) > 0.05
cmp_ = pd.Series({"normal workdays": workdays[~heavy].mean(), "holiday-heavy workdays": workdays[heavy].mean()})
ax = cmp_.plot.bar(color=["tab:blue", "tab:orange"], figsize=(5.5, 3.6), rot=0)
ax.set_ylabel("tickets / day"); ax.set_title("Workday volume: holiday vs normal")
plt.show()
print(f"normal: {cmp_.iloc[0]:.1f} tickets/day (n={int((~heavy).sum())})  |  holiday-heavy: {cmp_.iloc[1]:.1f} (n={int(heavy.sum())})")
print("country weights come from tickets BEFORE the test period (no leakage):")
display(country_weights.head(6).round(3).to_frame("share of tickets").T)

In [ ]:
# 2f. noise and spikes
wd_recent = workdays["2021-01-01":]
disp = wd_recent.var() / wd_recent.mean()
print(f"workday variance/mean since 2021: {disp:.1f}   (pure Poisson noise would be 1.0;")
print("this includes trend + holiday variation, so it is an upper bound on the true noise)\n")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(workdays.values, bins=40, color="tab:blue"); axes[0].set_title("Distribution of workday counts")
axes[0].set_xlabel("tickets / day")
top = daily.sort_values(ascending=False).head(10)
axes[1].bar([d.strftime("%Y-%m-%d") for d in top.index], top.values, color="tab:red")
axes[1].set_title("10 busiest days"); axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

**What this tells us**

* A strong **weekly cycle** (Sun–Thu busy, Fri/Sat quiet) — every model must capture it; the plain moving average cannot.
* **Holidays** reduce volume on working days, so the holiday feature is worth keeping.
* Daily counts are **noisy and occasionally spike far above normal.** Spikes cannot be predicted from past counts, which sets a floor on achievable daily accuracy.

## 3. Evaluation design

```
|------------ CV region (tuning + model selection) ------------|-- FINAL TEST --|
| train ... | fold1 | fold2 | fold3 | fold4 | fold5 | fold6    |  last 12 weeks  |
```

* **Walk-forward, expanding window.** At every weekly forecast origin the model is re-fitted on data *before* that origin and predicts the next 7 days.
* **6 folds × 8 weeks** are used for hyper-parameter tuning and for choosing the model.
* The **final test** is never used for tuning or selection. It is only looked at once.
* **Leakage guards:** lags/rolling stats are shifted by ≥ 7 days; holiday country weights use only pre-test tickets; a runtime assertion fails if a future lag feature contains NaN (= would have leaked).

In [ ]:
folds = make_folds(len(daily), FINAL_TEST, N_FOLDS, FOLD_DAYS)
rows = [{"fold": i, "from": daily.index[a].date(), "to": daily.index[b-1].date()} for i, (a, b) in enumerate(folds, 1)]
rows.append({"fold": "FINAL TEST", "from": daily.index[-FINAL_TEST].date(), "to": daily.index[-1].date()})
print(f"training data starts {daily.index[0].date()}")
display(pd.DataFrame(rows).set_index("fold"))

## 4. Models compared

| Group | Model | Idea |
|---|---|---|
| Baseline | `moving_avg_7`, `moving_avg_28` | flat forecast = recent average |
| Baseline | `seasonal_naive` | next Sunday = last Sunday |
| Baseline | `seasonal_moving_avg_4w` | next Sunday = mean of the last 4 Sundays (**the baseline to beat**) |
| Candidate | `holt_winters_tuned` | exponential smoothing with weekly seasonality |
| Candidate | `prophet_tuned` | trend + weekly seasonality + holiday regressor |
| Candidate | `xgboost_tuned` | lag, rolling, calendar and holiday features; average of the 5 best tuned configs |
| Candidate | `ensemble_prophet+xgb` | average of Prophet and XGBoost |

Tuning: random/grid search scored by **mean cross-validated MAE** (see `ml/artifacts/forecasting/tuning_trials_*.csv`).

In [ ]:
need = [OUT / f for f in ("model_comparison.csv", "cv_fold_scores.csv", "surge_eval.csv", "forecast_next_14d.csv", "best_params.json")]
missing = [str(p) for p in need if not p.exists()]
assert not missing, f"Run `python -m ml.forecasting.run_models` first. Missing: {missing}"

table = pd.read_csv(OUT / "model_comparison.csv")
folds_df = pd.read_csv(OUT / "cv_fold_scores.csv")
best = json.load(open(OUT / "best_params.json"))
selected = table.loc[0, "model"]

cols = ["model", "CV_MAE", "CV_MAE_std", "CV_weekly_WAPE_%", "TEST_MAE", "TEST_RMSE", "TEST_weekly_WAPE_%"]
display(table[cols].round(2))
print("Selected model (lowest CV MAE, NOT chosen on the test set):", selected)

In [ ]:
base = table.set_index("model").loc["seasonal_moving_avg_4w"]
sel = table.set_index("model").loc[selected]
print(f"CV   MAE: {sel.CV_MAE:.2f} vs baseline {base.CV_MAE:.2f}  ->  {100*(1-sel.CV_MAE/base.CV_MAE):.0f}% lower")
print(f"TEST MAE: {sel.TEST_MAE:.2f} vs baseline {base.TEST_MAE:.2f}  ->  {100*(1-sel.TEST_MAE/base.TEST_MAE):.0f}% lower  (only 12 weeks: noisy)")
print(f"Weekly-total error (WAPE): CV {sel['CV_weekly_WAPE_%']:.1f}% vs {base['CV_weekly_WAPE_%']:.1f}%  |  TEST {sel['TEST_weekly_WAPE_%']:.1f}% vs {base['TEST_weekly_WAPE_%']:.1f}%")

show = ["moving_avg_7", "seasonal_naive", "seasonal_moving_avg_4w", "holt_winters_tuned", "prophet_tuned", "xgboost_tuned", selected]
show = list(dict.fromkeys(show))
t = table.set_index("model").loc[show].iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(t.index, t["CV_MAE"], xerr=t["CV_MAE_std"], color="tab:blue", alpha=.85, capsize=3, label="CV MAE (mean ± std over folds)")
ax.scatter(t["TEST_MAE"], t.index, color="tab:orange", zorder=3, label="final-test MAE")
ax.set_xlabel("MAE (tickets / day) — lower is better"); ax.legend(); ax.set_title("Cross-validated vs final-test error")
plt.show()

In [ ]:
# consistency across folds: does the selected model beat the baseline in every fold, or only on average?
pv = folds_df.pivot(index="fold", columns="model", values="MAE")
cmp_models = ["seasonal_moving_avg_4w", "prophet_tuned", "xgboost_tuned", selected]
cmp_models = list(dict.fromkeys(cmp_models))
ax = pv[cmp_models].plot(marker="o", figsize=(9, 3.8)); ax.set_ylabel("MAE"); ax.set_title("MAE per CV fold")
plt.show()
wins = int((pv[selected] < pv["seasonal_moving_avg_4w"]).sum())
print(f"{selected} beats the seasonal baseline in {wins} of {len(pv)} folds")

## 5. Error analysis (final test period)

We re-create the selected model's 7-day-ahead predictions for the final test to see **where** it is wrong.
(The interval step re-runs cross-validation once, so this cell takes a minute or two.)

In [ ]:
def build_selected(horizon=HORIZON):
    pb = {k: v for k, v in best["prophet"].items() if k != "use_holidays"}
    pro = prophet_model(hs if best["prophet"]["use_holidays"] else None, **pb)
    if "xgboost_top5" in best:
        xs = []
        for c in best["xgboost_top5"]:
            c = dict(c); use_hs = c.pop("use_holidays")
            xs.append(xgboost_model(hs if use_hs else None, horizon, **c))
        xgb = ensemble(xs)
    else:
        c = {k: v for k, v in best["xgboost"].items() if k != "use_holidays"}
        xgb = xgboost_model(hs if best["xgboost"]["use_holidays"] else None, horizon, **c)
    return ensemble([pro, xgb])

assert selected == "ensemble_prophet+xgb", "This cell rebuilds the Prophet+XGB ensemble; adapt it if another model was selected."
fn = build_selected()
n = len(daily)
tp = walk_forward(daily, fn, n - FINAL_TEST, n, HORIZON)
tp["error"] = tp["actual"] - tp["forecast"]
tp["abs_error"] = tp["error"].abs()
tp["weekday"] = tp["date"].dt.day_name()

# prediction interval: empirical 5%/95% residual quantiles from cross-validation, split workday / weekend
cvp, _ = cross_validate(daily, fn, folds, HORIZON)
cvp["wk"] = np.where(cvp["date"].dt.dayofweek.isin([4, 5]), "weekend", "workday")
q = (cvp["actual"] - cvp["forecast"]).groupby(cvp["wk"]).quantile([.05, .95]).unstack()
wk = np.where(tp["date"].dt.dayofweek.isin([4, 5]), "weekend", "workday")
tp["lower"] = np.clip(tp["forecast"] + q.loc[wk, 0.05].to_numpy(), 0, None)
tp["upper"] = tp["forecast"] + q.loc[wk, 0.95].to_numpy()
cover = ((tp["actual"] >= tp["lower"]) & (tp["actual"] <= tp["upper"])).mean()

basep = walk_forward(daily, BASELINES["seasonal_moving_avg_4w"], n - FINAL_TEST, n, HORIZON)
thr = float(workdays[:test_start - pd.Timedelta(days=1)].iloc[-365:].quantile(.75))

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between(tp["date"], tp["lower"], tp["upper"], color="tab:blue", alpha=.15, label="90% interval")
ax.plot(daily.index[-FINAL_TEST:], daily.iloc[-FINAL_TEST:], color="black", lw=1.4, label="actual")
ax.plot(tp["date"], tp["forecast"], color="tab:blue", lw=1.4, label=f"{selected} (7-day-ahead)")
ax.plot(basep["date"], basep["forecast"], color="tab:orange", lw=1, ls="--", label="seasonal moving avg (baseline)")
ax.axhline(thr, color="tab:red", lw=.8, ls=":", label=f"high-load threshold ({thr:.0f})")
ax.set_title("Final test (never used for tuning)"); ax.set_ylabel("tickets / day"); ax.legend(ncol=2, fontsize=9)
plt.show()
print(f"90% interval coverage on final test: {100*cover:.1f}%  (target 90%)")

In [ ]:
print(f"mean error (actual - forecast): {tp['error'].mean():+.2f}  (positive = model under-forecasts)\n")

worst = tp.sort_values("abs_error", ascending=False).head(8)[["date", "weekday", "actual", "forecast", "error"]].round(1)
worst["date"] = worst["date"].dt.date
display(worst.reset_index(drop=True))

srt = tp["abs_error"].sort_values(ascending=False)
for k in (2, 5, 10):
    print(f"top {k:>2} worst days = {100*srt.head(k).sum()/srt.sum():.0f}% of total absolute error")

by_wd = tp.groupby("weekday")["abs_error"].mean().reindex(["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"])
ax = by_wd.plot.bar(color="tab:blue", figsize=(8, 3.4), rot=0); ax.set_ylabel("MAE"); ax.set_xlabel("")
ax.set_title("Error by weekday (final test)"); plt.show()

## 6. High-load flag and the next forecast

`high_load_flag = 1` when the forecast for a day reaches the **75th percentile of the last 365 workdays**.
Its quality is measured on the final test: precision = how many flagged days were truly heavy, recall = how many truly heavy days were flagged.

In [ ]:
display(pd.read_csv(OUT / "surge_eval.csv"))
print("Reading: a high threshold flags almost nothing (low recall). True spikes are not predictable from history.")
print("Use the flag as a planning hint, not as an alarm.")

In [ ]:
f14 = pd.read_csv(OUT / "forecast_next_14d.csv", parse_dates=["date"])
fig, ax = plt.subplots(figsize=(11, 3.8))
colors = np.where(f14["high_load_flag"] == 1, "tab:red", "tab:blue")
ax.bar(f14["date"], f14["forecast"], color=colors, alpha=.85)
ax.errorbar(f14["date"], f14["forecast"], yerr=[f14["forecast"] - f14["lower_90"], f14["upper_90"] - f14["forecast"]],
            fmt="none", ecolor="black", capsize=3, lw=1)
ax.axhline(f14["high_load_threshold"].iloc[0], color="tab:red", ls=":", lw=1)
ax.set_title(f"Forecast for the 14 days after the data ends ({daily.index[-1].date()}) — red = high-load day")
ax.set_ylabel("tickets / day"); plt.xticks(rotation=45); plt.show()
display(f14[["date", "weekday", "forecast", "lower_90", "upper_90", "high_load_flag"]].assign(date=lambda d: d["date"].dt.date))
print("Accuracy was validated for the 7-day horizon only; days 8-14 are less certain.")

## 7. Conclusions and limitations

**What works**

* A tuned Prophet + XGBoost ensemble beats the strongest simple baseline (weekday-aware moving average) in cross-validation, and the gain is visible fold by fold (section 4).
* The holiday feature (built from the masked country prefix of the project code) carries real signal.

**What does not (be explicit about this in the report)**

* **Spikes are not predicted.** A handful of extreme days account for a large share of the final-test error (section 5).
* **Weekly totals:** on the final test the ensemble is **not clearly better** than the baseline, even though it was better in CV.
* **The final test is only 12 weeks long**, so its numbers are noisy. Cross-validation is the stronger evidence.
* **Intervals are slightly too narrow** (coverage below 90%).
* **The high-load flag is a weak signal** and flips when forecasts sit close to the threshold.
* **Dataset ends in March 2023.** The "next 14 days" forecast is a demonstration of the pipeline, not a live forecast.

**Next steps**

1. Weekly-level forecast (what managers actually plan on).
2. Calibrated prediction intervals and a tiered load flag (normal / high / very high).
3. Forecasts per country or project, to see whether spikes come from one source.
4. Hand the forecast table to the optimiser (see `docs/model3_workload_forecasting_report.md`).